# 01 — Data Pull, Inspect & Validate (Sample Databases)

**Thesis:** Implied Volatility Smile Spillovers (AP-33)  
**Author:** Başar Hacımustafaoğlu — 1******6  
**Purpose:** Pull a small sample from each accessible thesis-relevant database, inspect the structure, and validate the data quality.  

---

## What this notebook does

1. Connects to WRDS
2. For each accessible database: pulls a small sample (top 500 rows)
3. Runs full validation on each pull (shape, dtypes, missingness, date coverage, duplicates)
4. Saves raw pulls to `data/raw/` as CSV
5. Saves a validation summary report to `logs/`

**This notebook does NOT clean, merge, or analyse anything.**  
**It only answers: what does this data look like, and is it usable?**

---

> ⚠️ **Run notebook 00 first** to verify your WRDS connection.

## Step 1 — Imports and connection

In [1]:
import os
import sys
import datetime
import json
import pandas as pd
import wrds
from dotenv import load_dotenv

sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from wrds_utils import connect_wrds, validate_df

load_dotenv()
conn = connect_wrds()

TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RAW_DIR   = "../data/raw"
LOG_DIR   = "../logs"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f"Run timestamp: {TIMESTAMP}")

Connecting to WRDS as: basar
Loading library list...
Done
Connection established.
Run timestamp: 20260314_142408


## Step 2 — Define pull targets

For each database we have access to, we specify which table to pull a sample from and why.

In [2]:
# Each entry: (library, table, description, n_rows_to_pull)
# We pull only 500 rows per table — this is a structural inspection, not a data download

PULL_TARGETS = [

    # OptionMetrics US sample (2014)
    ("optionmsamp_us",     "vsurfd2014",              "OptionMetrics US — volatility surface (2014)",        500),
    ("optionmsamp_us",     "opprcd2014",              "OptionMetrics US — raw option prices (2014)",         500),
    ("optionmsamp_us",     "secnmd",                  "OptionMetrics US — security names",                   500),
    ("optionmsamp_us",     "secprd",                  "OptionMetrics US — underlying prices",                500),

    # OptionMetrics Europe sample (2013)
    ("optionmsamp_europe", "volatility_surface_2013", "OptionMetrics EU — volatility surface (2013)",        500),
    ("optionmsamp_europe", "option_price_2013",       "OptionMetrics EU — raw option prices (2013)",         500),
    ("optionmsamp_europe", "security_name",           "OptionMetrics EU — security names",                   500),
    ("optionmsamp_europe", "security_price",          "OptionMetrics EU — underlying prices",                500),
    ("optionmsamp_europe", "historical_volatility",   "OptionMetrics EU — historical volatility",            500),

    # CBOE
    ("cboe",               "cboe",                    "CBOE — VIX and volatility indices",                   500),

    # FRB
    ("frb",                "rates_daily",             "FRB — daily interest rates",                          500),

    # CRSP
    ("crsp",               "dsf",                     "CRSP — daily stock file",                             500),
    ("crsp",               "dsp500",                  "CRSP — daily S&P 500 index",                          500),
    ("crsp",               "dsi",                     "CRSP — daily market index",                           500),

    # Compustat Global
    ("comp",               "g_secd",                  "Compustat Global — daily security prices",            500),
    ("comp",               "g_idx_daily",             "Compustat Global — daily index prices",               500),
    ("comp",               "g_exrt_dly",              "Compustat Global — daily exchange rates",             500),

    # Fama-French
    ("ff",                 "factors_daily",           "Fama-French — daily factors",                         500),


    # WRDS Apps
    ("wrdsapps",           "eushort",                 "WRDS Apps — EU short sales",                          500),
    ("wrdsapps",           "intl_market_returns",     "WRDS Apps — international market returns",            500),

    # MacroFin
    ("macrofin",           "q_factors_daily",         "MacroFin — Q-factors daily",                          500),
]

print(f"Pull targets defined: {len(PULL_TARGETS)} tables")

Pull targets defined: 21 tables


## Step 3 — Pull and validate each table

For each target we:
1. Pull `n` rows
2. Run validation
3. Save raw CSV to `data/raw/`
4. Log the result

Errors are caught and logged — a failed pull does not stop the notebook.

In [3]:
# ================================================================
# STEP 3 — PULL DATA WITH DATE FILTER (MODERN PERIOD)
# Pulls 500 most recent rows per table after 2010-01-01
# For inspection only — full pull happens in notebook 02
# ================================================================

pull_log   = []
pulled_dfs = {}

# Tables that have a date column — we filter to modern period
DATE_FILTERED = {
    "optionmsamp_us.vsurfd2014":                  "date",
    "optionmsamp_us.opprcd2014":                  "date",
    "optionmsamp_us.secprd":                      "date",
    "optionmsamp_europe.volatility_surface_2013": "date",
    "optionmsamp_europe.option_price_2013":       "date",
    "optionmsamp_europe.security_price":          "date",
    "optionmsamp_europe.historical_volatility":   "date",
    "cboe.cboe":                                  "date",
    "frb.rates_daily":                            "date",
    "crsp.dsi":                                   "date",
    "crsp.dsf":                                   "date",
    "crsp.dsp500":                                "caldt", 
    "comp.g_secd":                                "datadate", 
    "comp.g_idx_daily":                           "datadate", 
    "comp.g_exrt_dly":                            "datadate", 
    "ff.factors_daily":                           "date",
    "djones.djdaily":                             "date",
    "phlx.iv":                                    "qdate",
    "wrdsapps.eushort":                           "position_date",
    "macrofin.q_factors_daily":                   "date",
}

for library, table, description, n_rows in PULL_TARGETS:
    key = f"{library}.{table}"

    print(f"\n{'='*60}")
    print(f"Pulling: {key}")
    print(f"{'='*60}")

    log_entry = {
        "library":     library,
        "table":       table,
        "description": description,
        "n_requested": n_rows,
        "status":      None,
        "n_rows":      None,
        "n_cols":      None,
        "columns":     None,
        "error":       None,
        "saved_to":    None,
    }

    try:
        if key in DATE_FILTERED:
            date_col = DATE_FILTERED[key]
            if key == "comp.g_secd":
                sql = f"""
                    SELECT * FROM {library}.{table}
                    WHERE {date_col} >= '2010-01-01'
                    AND {date_col} <= '2024-12-31'
                    AND prccd IS NOT NULL
                    ORDER BY {date_col} DESC
                    LIMIT {n_rows}
                """
            else:
                sql = f"""
                    SELECT * FROM {library}.{table}
                    WHERE {date_col} >= '2010-01-01'
                    ORDER BY {date_col} DESC
                    LIMIT {n_rows}
                """
        else:
            sql = f"""
                SELECT * FROM {library}.{table}
                LIMIT {n_rows}
            """

        df = conn.raw_sql(sql)

        log_entry["status"]  = "SUCCESS"
        log_entry["n_rows"]  = len(df)
        log_entry["n_cols"]  = len(df.columns)
        log_entry["columns"] = list(df.columns)

        validate_df(df, key)

        filename = f"{library}__{table}__MODERN_{TIMESTAMP}.csv"
        filepath = os.path.join(RAW_DIR, filename)
        df.to_csv(filepath, index=False)
        log_entry["saved_to"] = filepath
        print(f"✓ Saved to: {filepath}")

        pulled_dfs[key] = df

    except Exception as e:
        log_entry["status"] = "ERROR"
        log_entry["error"]  = str(e)
        print(f"✗ ERROR: {e}")
        print("  Skipping and continuing.")

    pull_log.append(log_entry)

print(f"\n{'='*60}")
print("All pull attempts complete.")


Pulling: optionmsamp_us.vsurfd2014

VALIDATION REPORT: optionmsamp_us.vsurfd2014

[1] Shape: 500 rows x 9 columns

[2] Columns and dtypes:
    secid                               Float64
    date                                string
    days                                Float64
    delta                               Float64
    impl_volatility                     Float64
    impl_strike                         Float64
    impl_premium                        Float64
    dispersion                          Float64
    cp_flag                             string

[3] Missingness:
    No missing values detected.

[4] Date coverage:
    date: 2014-03-13 00:00:00 → 2014-03-14 00:00:00

[5] Duplicate rows: 0


✓ Saved to: ../data/raw/optionmsamp_us__vsurfd2014__MODERN_20260314_142408.csv

Pulling: optionmsamp_us.opprcd2014

VALIDATION REPORT: optionmsamp_us.opprcd2014

[1] Shape: 500 rows x 22 columns

[2] Columns and dtypes:
    secid                               Float64
    date       

In [4]:
# Quick row name check for pull date fix
df = conn.raw_sql("""
    SELECT gvkey, datadate, conm, prccd, curcdd, loc
    FROM comp.g_secd
    WHERE datadate BETWEEN '2010-01-01' AND '2024-12-31'
    AND prccd IS NOT NULL
    ORDER BY datadate DESC
    LIMIT 10
""")
print(df.to_string())

    gvkey    datadate                          conm    prccd curcdd  loc
0  371311  2024-12-31  INTERCAPITAL KROVNI UCITS ET    10.15    EUR  HRV
1  371169  2024-12-31  BANDHAN MUTUAL FUND - BANDHA   257.03    INR  IND
2  371125  2024-12-31            NOBLE POLYMERS LTD     0.31    INR  IND
3  371062  2024-12-31     IPA SECURITIES INVESTMENT   8500.0    VND  VNM
4  371026  2024-12-31   KSM MUTUAL FUNDS LTD. - KSM  42.2507    ILS  ISR
5  371025  2024-12-31   MEITAV TACHLIT MUTUAL FUNDS    4.719    ILS  ISR
6  371014  2024-12-31    KSM MUTUAL FUNDS LTD - KSM    82.99    ILS  ISR
7  371013  2024-12-31   KSM MUTUAL FUNDS LTD. - KSM  41.7482    ILS  ISR
8  371012  2024-12-31      INMCAMSAL GESTION SIL SA   1.0038    EUR  ESP
9  371011  2024-12-31       CHINA ASSET MGMT CO LTD     1.06    CNY  CHN


## Step 4 — Pull summary

Quick overview of what succeeded, what failed, and how many rows we got from each table.

In [5]:
print(f"Pull summary — {TIMESTAMP}\n")
print(f"{'Table':<50} {'Status':<10} {'Rows':>6} {'Cols':>5}")
print("-" * 75)

n_success = 0
n_error   = 0

for entry in pull_log:
    key    = f"{entry['library']}.{entry['table']}"
    status = entry['status']
    rows   = entry['n_rows'] if entry['n_rows'] is not None else '-'
    cols   = entry['n_cols'] if entry['n_cols'] is not None else '-'
    print(f"  {key:<48} {status:<10} {str(rows):>6} {str(cols):>5}")
    if status == 'SUCCESS':
        n_success += 1
    else:
        n_error += 1

print("-" * 75)
print(f"  Succeeded: {n_success} | Errors: {n_error} | Total attempted: {len(pull_log)}")

if n_error > 0:
    print("\nFailed tables:")
    for entry in pull_log:
        if entry['status'] == 'ERROR':
            print(f"  {entry['library']}.{entry['table']}: {entry['error'][:120]}")

Pull summary — 20260314_142408

Table                                              Status       Rows  Cols
---------------------------------------------------------------------------
  optionmsamp_us.vsurfd2014                        SUCCESS       500     9
  optionmsamp_us.opprcd2014                        SUCCESS       500    22
  optionmsamp_us.secnmd                            SUCCESS         3     8
  optionmsamp_us.secprd                            SUCCESS        10    11
  optionmsamp_europe.volatility_surface_2013       SUCCESS       500    10
  optionmsamp_europe.option_price_2013             SUCCESS       500    24
  optionmsamp_europe.security_name                 SUCCESS         1     6
  optionmsamp_europe.security_price                SUCCESS        50    14
  optionmsamp_europe.historical_volatility         SUCCESS       143     5
  cboe.cboe                                        SUCCESS       500    17
  frb.rates_daily                                  SUCCESS       50

## Step 5 — Interactive inspection

Look at the most thesis-relevant tables in more detail — the volatility surfaces and the security names.
These cells are for eyeballing the data, not for production use.

In [6]:
# EU volatility surface — check columns, delta range, days range
key = "optionmsamp_europe.volatility_surface_2013"
if key in pulled_dfs:
    df = pulled_dfs[key]
    print(f"EU volatility surface — {len(df)} rows\n")
    print("Column names:", list(df.columns))
    print(f"\nDelta range : {df['delta'].min()} → {df['delta'].max()}")
    print(f"Days range  : {df['days'].min()} → {df['days'].max()}")
    print(f"Call/put    : {df['callput'].unique()}")
    print(f"Date range  : {df['date'].min()} → {df['date'].max()}")
    print(f"\nFirst 5 rows:")
    print(df.head().to_string())
else:
    print(f"Table not loaded: {key}")

EU volatility surface — 500 rows

Column names: ['securityid', 'days', 'delta', 'callput', 'impliedvol', 'strike', 'premium', 'dispersion', 'currency', 'date']

Delta range : -80 → 80
Days range  : 30.0 → 730.0
Call/put    : <StringArray>
['P', 'C']
Length: 2, dtype: string
Date range  : 2013-03-14 → 2013-03-15

First 5 rows:
   securityid   days  delta callput  impliedvol     strike   premium  dispersion  currency        date
0    500096.0  122.0    -55       P    0.229267  80.988411  5.409452    0.001412     814.0  2013-03-15
1    500096.0  730.0    -20       P    0.257929   62.00779  3.933668    0.009449     814.0  2013-03-15
2    500096.0  273.0    -60       P    0.227319  84.830963  9.777775    0.001917     814.0  2013-03-15
3    500096.0  152.0    -30       P    0.239527  73.692596  2.572347    0.005472     814.0  2013-03-15
4    500096.0   60.0    -55       P    0.231189  79.891678   3.73528     0.00643     814.0  2013-03-15


In [7]:
# US volatility surface — note: column names differ from EU
# US uses 'impl_volatility', 'cp_flag', 'secid' vs EU's 'impliedvol', 'callput', 'securityid'
key = "optionmsamp_us.vsurfd2014"
if key in pulled_dfs:
    df = pulled_dfs[key]
    print(f"US volatility surface — {len(df)} rows\n")
    print("Column names:", list(df.columns))
    print(f"\nDelta range : {df['delta'].min()} → {df['delta'].max()}")
    print(f"Days range  : {df['days'].min()} → {df['days'].max()}")
    print(f"Call/put    : {df['cp_flag'].unique()}")
    print(f"Date range  : {df['date'].min()} → {df['date'].max()}")
    print(f"\nFirst 5 rows:")
    print(df.head().to_string())
else:
    print(f"Table not loaded: {key}")

US volatility surface — 500 rows

Column names: ['secid', 'date', 'days', 'delta', 'impl_volatility', 'impl_strike', 'impl_premium', 'dispersion', 'cp_flag']

Delta range : -80.0 → 80.0
Days range  : 30.0 → 730.0
Call/put    : <StringArray>
['P', 'C']
Length: 2, dtype: string
Date range  : 2014-03-13 → 2014-03-14

First 5 rows:
      secid        date   days  delta  impl_volatility  impl_strike  impl_premium  dispersion cp_flag
0  101594.0  2014-03-14  152.0  -20.0         0.246302     459.6892      10.10119    0.009491       P
1  101594.0  2014-03-14  730.0  -25.0         0.267833     426.1938      37.01022     0.00241       P
2  101594.0  2014-03-14  365.0   35.0         0.245204      583.174      26.43184    0.002982       C
3  101594.0  2014-03-14  152.0  -25.0         0.242656      472.713      13.34747    0.006066       P
4  101594.0  2014-03-14  152.0   35.0         0.234488     558.9031       17.0682    0.003466       C


In [8]:
# Security names — confirm which securities are in each sample
for key in ["optionmsamp_europe.security_name", "optionmsamp_us.secnmd"]:
    if key in pulled_dfs:
        print(f"{key}:")
        print(pulled_dfs[key].to_string())
        print()

optionmsamp_europe.security_name:
   securityid       valor     issuer    sedol          isin effectivedate
0    500096.0  11730015.0  adidas AG  4031976  DE000A1EWWW0    2010-10-11

optionmsamp_us.secnmd:
      secid effect_date     cusip ticker class              issuer issue   sic
0  101594.0  1996-01-02  03783310   AAPL  <NA>  APPLE COMPUTER INC   COM  <NA>
1  101594.0  2000-11-28  03783310   AAPL  <NA>  APPLE COMPUTER INC   COM  3571
2  101594.0  2007-01-11  03783310   AAPL  <NA>           APPLE INC   COM  3571



In [9]:
# VIX — check available columns and date coverage
key = "cboe.cboe"
if key in pulled_dfs:
    df = pulled_dfs[key]
    print(f"CBOE — {len(df)} rows")
    print(f"Columns: {list(df.columns)}")
    print(f"Date range: {df['date'].min()} → {df['date'].max()}")
    print(f"VIX range : {df['vix'].min()} → {df['vix'].max()}")
    print(f"\nSample:")
    print(df[['date', 'vix']].head(10).to_string(index=False))

CBOE — 500 rows
Columns: ['date', 'vixo', 'vixh', 'vixl', 'vix', 'vxoo', 'vxoh', 'vxol', 'vxo', 'vxno', 'vxnh', 'vxnl', 'vxn', 'vxdo', 'vxdh', 'vxdl', 'vxd']
Date range: 2024-03-22 → 2026-02-27
VIX range : 11.86 → 52.33

Sample:
      date    vix
2026-02-27  19.86
2026-02-26  18.63
2026-02-25  17.93
2026-02-24  19.55
2026-02-23  21.01
2026-02-20  19.09
2026-02-19  20.23
2026-02-18  19.62
2026-02-17  20.29
2026-02-16   21.2


## Step 6 — Save full log

Save the complete pull log as JSON so we have a permanent audit trail of what was pulled, when, how many rows, and any errors.

In [10]:
log_path = os.path.join(LOG_DIR, f"01_pull_log__{TIMESTAMP}.json")

with open(log_path, 'w') as f:
    json.dump(pull_log, f, indent=2, default=str)

print(f"Pull log saved to: {log_path}")
print(f"Entries: {len(pull_log)}")

# Also print a quick readable version
print("\nLog preview (first 3 entries):")
for entry in pull_log[:3]:
    print(f"  {entry['library']}.{entry['table']}: {entry['status']}, {entry['n_rows']} rows")

Pull log saved to: ../logs/01_pull_log__20260314_142408.json
Entries: 21

Log preview (first 3 entries):
  optionmsamp_us.vsurfd2014: SUCCESS, 500 rows
  optionmsamp_us.opprcd2014: SUCCESS, 500 rows
  optionmsamp_us.secnmd: SUCCESS, 3 rows


## Step 7 — Close connection

Always close the WRDS connection when done — leaving it open wastes resources.

In [11]:
conn.close()
print("WRDS connection closed.")
print(f"\nNotebook 01 complete — {TIMESTAMP}")
print("Raw CSVs saved to: ../data/raw/")
print("Pull log saved to: ../logs/")
print("\nNext: run notebook 02 to pull full window data.")

WRDS connection closed.

Notebook 01 complete — 20260314_142408
Raw CSVs saved to: ../data/raw/
Pull log saved to: ../logs/

Next: run notebook 02 to pull full window data.
